# 03 — CV de Shannon intra-partido

Umbral AgendaPP: **CV ≤ 0.3 → uniformidad (H1a)**, **CV > 0.3 → autonomía individual (H2a)**.


In [ ]:
# Cargar datos: cambia URL o XLSX segun corresponda
from pathlib import Path
import pandas as pd, numpy as np
from agendapp.io import fetch_endpoint, load_xlsx_municipio
from agendapp.transform import matriz_concejal_tema, binarizar, perfil_partido, filtrar_min_instrumentos
from agendapp.indices import shannon_norm, cv_shannon, jaccard_pairwise_mean, party_correlation
from agendapp import viz

# Opcion A: endpoint Apps Script (descomentar y poner URL real)
# URL = "https://script.google.com/macros/s/AKfycb.../exec"
# data = fetch_endpoint(URL)
# df_inst = pd.DataFrame(data["instrumentos"])

# Opcion B: xlsx local (piloto Guarne)
XLSX = Path("../..") / "Guarne_DILIGENCIADO.xlsx"
d = load_xlsx_municipio(XLSX)
df_inst = d["instrumentos"]
df_inst["Partido / Movimiento"] = df_inst["Partido / Movimiento"].astype(str).str.strip().str.upper()
df_inst.shape


In [ ]:
M = matriz_concejal_tema(df_inst, col_tema='Tematica', roles=('Proponente',))
h = M.apply(shannon_norm, axis=1)
asign = (df_inst[df_inst['Rol'].eq('Proponente')]
    .groupby('ID_Concejal')['Partido / Movimiento']
    .agg(lambda s: s.value_counts().idxmax()))
tabla = pd.DataFrame({'H': h, 'partido': asign})
cv = tabla.groupby('partido')['H'].apply(lambda x: cv_shannon(x.values))
cv.dropna()


## Barras (verde = uniformidad H1a)


In [ ]:
viz.barras_cv_por_partido(cv.dropna())
